# Task 5 – Semantic Clustering of SMS Messages

## Why embeddings instead of TF-IDF?

TF-IDF treats every word as an independent symbol. Two semantically identical spam
messages using different vocabulary land in completely different positions in the
TF-IDF vector space.

**Embeddings** encode meaning as dense vectors trained on billions of texts.
Semantically similar messages → nearby vectors → same cluster.

## Provider strategy — diversity over redundancy

Three embedding providers are used, each from a different vendor and model family:

| Provider | Model | Dimensions | Focus |
|----------|-------|-----------|-------|
| Ollama (local) | nomic-embed-text | 768 | English, privacy-first, no API key |
| OpenAI | text-embedding-3-small | 1536 | English, state-of-the-art |
| Azure AI Foundry | Cohere-embed-v3-multilingual | 1024 | **Multilingual** — relevant for German SMS |

`text-embedding-3-small` on Azure would also be possible (it's in the catalog) but
would require creating an explicit Azure deployment and would duplicate the OpenAI
provider without adding value. Cohere multilingual is available as a serverless model
and is the more interesting choice for A1's actual use case.

## Architecture

The clustering logic depends only on the abstract `EmbeddingProvider` interface.
Swap the backend by changing one variable here or via the CLI flag — zero code changes:

```bash
sms-analyze clustering --provider ollama    # local, no API key
sms-analyze clustering --provider openai    # SMS_OPENAI_API_KEY in .env
sms-analyze clustering --provider azure     # SMS_AZURE_OPENAI_* in .env
```

API keys and endpoints are secrets → `.env` file.

## Setup

For **Ollama** (default): run `scripts/setup_ollama.sh` once before this notebook.  
For **OpenAI / Azure AI**: put keys in `.env` — this notebook detects them automatically.

In [ ]:
import os
import random
from IPython.display import display
from pathlib import Path

from sms_analysis import load_dataset
from sms_analysis.config import Settings
from sms_analysis.clustering import SemanticClusterer
from sms_analysis.clustering.providers import OllamaProvider, OpenAIProvider, AzureOpenAIProvider
from sms_analysis.data.preprocessing import preprocess
from sms_analysis.visualization import plot_clusters_2d, save_figure

DATASET = Path("../assignment/3D8026C4.zip/SMSSpamCollection")
OUTPUTS = Path("outputs")

settings = Settings()   # reads secrets from .env
dataset  = load_dataset(DATASET)

# Fixed sample used by all providers for a fair comparison
random.seed(42)
n_sample = 500
n_spam   = int(n_sample * len(dataset.spam) / len(dataset))
n_ham    = n_sample - n_spam

sample_msgs   = random.sample(dataset.ham, n_ham) + random.sample(dataset.spam, n_spam)
random.shuffle(sample_msgs)
sample_texts  = [preprocess(m.text) for m in sample_msgs]
sample_labels = [m.label.value for m in sample_msgs]

dataset.to_dataframe()

In [ ]:
# Auto-detect available providers and run clustering on the same sample.
# Results are stored in `results` dict keyed by provider name.

results: dict = {}
PROVIDER_LABELS: dict[str, str] = {}

# ── Ollama (always attempted — no key required) ───────────────────────────────
try:
    provider = OllamaProvider(model=settings.ollama_model, base_url=settings.ollama_base_url)
    clusterer = SemanticClusterer(provider)
    r = clusterer.fit(sample_texts, k_range=range(3, 12))
    results["ollama"] = r
    PROVIDER_LABELS["ollama"] = f"Ollama ({settings.ollama_model})"
    print(f"✓ Ollama ({settings.ollama_model}): {r.summary()}")
except Exception as exc:
    print(f"✗ Ollama: {exc}")

# ── OpenAI (if SMS_OPENAI_API_KEY is set) ────────────────────────────────────
openai_key = settings.openai_api_key or os.environ.get("OPENAI_API_KEY", "")
if openai_key:
    try:
        provider = OpenAIProvider(api_key=openai_key, model=settings.openai_model)
        clusterer = SemanticClusterer(provider)
        r = clusterer.fit(sample_texts, k_range=range(3, 12))
        results["openai"] = r
        PROVIDER_LABELS["openai"] = f"OpenAI ({settings.openai_model})"
        print(f"✓ OpenAI ({settings.openai_model}): {r.summary()}")
    except Exception as exc:
        print(f"✗ OpenAI: {exc}")
else:
    print("– OpenAI: no key  (set SMS_OPENAI_API_KEY in .env to enable)")

# ── Azure AI Foundry (if SMS_AZURE_OPENAI_* are set) ─────────────────────────
azure_key      = settings.azure_openai_api_key
azure_endpoint = settings.azure_openai_endpoint
if azure_key and azure_endpoint:
    try:
        provider = AzureOpenAIProvider(
            api_key=azure_key,
            endpoint=azure_endpoint,
            model=settings.azure_openai_model,
        )
        clusterer = SemanticClusterer(provider)
        r = clusterer.fit(sample_texts, k_range=range(3, 12))
        results["azure"] = r
        PROVIDER_LABELS["azure"] = f"Azure AI ({settings.azure_openai_model})"
        print(f"✓ Azure AI ({settings.azure_openai_model}): {r.summary()}")
    except Exception as exc:
        print(f"✗ Azure AI: {exc}")
else:
    print("– Azure AI: no credentials  (set SMS_AZURE_OPENAI_API_KEY + SMS_AZURE_OPENAI_ENDPOINT in .env to enable)")

print(f"\nProviders completed: {list(results)}")

In [ ]:
# Provider comparison table
import pandas as pd
import numpy as np

rows = []
for pname, r in results.items():
    sizes = [int(np.sum(r.labels == k)) for k in range(r.n_clusters)]
    rows.append({
        "Provider":    PROVIDER_LABELS[pname],
        "k":           r.n_clusters,
        "Silhouette":  round(r.silhouette, 3),
        "Cluster sizes": str(sorted(sizes, reverse=True)),
    })

df_cmp = pd.DataFrame(rows).set_index("Provider")
print("Provider comparison (same 500-message sample)\n")
print(df_cmp.to_string())

## Cluster visualisation (UMAP → 2D)

In [ ]:
for pname, r in results.items():
    fig = plot_clusters_2d(
        r.embeddings_2d,
        r.labels,
        sample_texts,
        title=f"Semantic Clusters — {PROVIDER_LABELS[pname]} (UMAP 2D)",
    )
    display(fig)
    save_figure(fig, OUTPUTS / f"05_semantic_clusters_{pname}.png")

## Elbow plot (inertia vs K)

In [ ]:
import matplotlib.pyplot as plt

for pname, r in results.items():
    if not r.inertias:
        continue
    fig, ax = plt.subplots(figsize=(8, 4))
    k_vals = list(range(3, 3 + len(r.inertias)))
    ax.plot(k_vals, r.inertias, marker="o", color="steelblue")
    ax.axvline(r.n_clusters, color="tomato", linestyle="--",
               label=f"Selected K={r.n_clusters}")
    ax.set_xlabel("K (number of clusters)")
    ax.set_ylabel("Inertia")
    ax.set_title(f"Elbow Plot — {PROVIDER_LABELS[pname]}", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    display(fig)
    save_figure(fig, OUTPUTS / f"05_elbow_plot_{pname}.png")

## Sample messages per cluster

In [ ]:
# Show sample messages for the first available result (Ollama baseline)
import numpy as np

first_key = list(results)[0]
r = results[first_key]
print(f"Provider: {PROVIDER_LABELS[first_key]}\n")

for k in range(r.n_clusters):
    indices = np.where(r.labels == k)[0]
    picks   = np.random.choice(indices, min(3, len(indices)), replace=False)
    print(f"── Cluster {k} (n={len(indices)}) ──")
    for i in picks:
        print(f"  [{sample_labels[i]:4}] {sample_texts[i][:100]}")
    print()

In [ ]:
# Spam / HAM composition per cluster — for each provider
import pandas as pd
import numpy as np

for pname, r in results.items():
    print(f"\n{'='*60}")
    print(f"Provider: {PROVIDER_LABELS[pname]}")
    print(f"{'='*60}")
    rows = []
    for k in range(r.n_clusters):
        indices = np.where(r.labels == k)[0]
        n_sp  = sum(1 for i in indices if sample_labels[i] == "spam")
        n_ha  = len(indices) - n_sp
        sp_pct = n_sp / len(indices) * 100
        rows.append({"Cluster": k, "Size": len(indices), "HAM": n_ha, "SPAM": n_sp, "SPAM %": round(sp_pct, 1)})
    df = pd.DataFrame(rows).set_index("Cluster")
    print(df.to_string())
    total = sum(r["SPAM"] for r in rows)
    print(f"Total SPAM in sample: {total} / {sum(r['Size'] for r in rows)}")

## Summary

### What K-Means found

Semantic clustering reveals **thematic groups** that TF-IDF-based analysis cannot surface — two messages using completely different words but expressing the same intent land in the same cluster.

**SPAM concentrates into 1–2 clusters** (visible in the cluster composition table above). Within those:
- One cluster captures **UK premium-rate fraud**: "Hard LIVE 121 chat", "call 09094…", shortcodes, per-minute pricing. These are structurally identical — repetitive call-to-action + premium number.
- Another captures **prize/lottery spam**: "won a £1000 prize", "GUARANTEED", "reply WIN". Different surface text, same campaign template.

**HAM spreads across multiple clusters**, reflecting the genuine diversity of human conversation:
- Short acknowledgements ("Ok", "K", "Yes")
- Location/logistics coordination
- Personal and emotional messages
- University-context Singlish ("lor", "lah")
- Casual English dialogue

The contrast is striking: **SPAM is semantically repetitive** (finite set of manipulation templates), **HAM is semantically diverse** (unlimited topics and registers). This is why even a simple linear model achieves very high AUC — the feature space is naturally separable.

### Provider comparison

| Provider | Embedding model | k | Silhouette | SPAM cluster |
|----------|----------------|---|-----------|--------------|
| Ollama (local) | nomic-embed-text | 4 | 0.022 | C3: 55% spam |
| OpenAI | text-embedding-3-small | 4 | 0.042 | C1: 51% spam |
| Azure AI Foundry | Cohere-embed-v3-multilingual | 3 | 0.044 | C2: 50% spam, C0/C1: <3% |

Key observations:
- **Silhouette (cluster quality)**: Azure/Cohere ≈ OpenAI > Ollama. Cloud embeddings produce tighter, better-separated clusters on this dataset.
- **Spam concentration**: Cohere multilingual finds a cleaner structure — ~91% of all spam lands in a single cluster while the other two clusters are nearly pure HAM. OpenAI and Ollama distribute spam more evenly.
- **Cluster count**: Cohere finds 3 clusters (a near-binary HAM/SPAM structure) whereas OpenAI and Ollama find 4 (further subdividing HAM into conversational subgroups).
- **Multilingual relevance**: For A1's actual use case (German SMS), the multilingual model is the best choice — it encodes German semantics instead of treating German words as noise.

### UMAP note

The 2D axes have no direct meaning — UMAP finds a low-dimensional layout that preserves neighbourhood structure from the high-dimensional embedding space. Only relative positions and cluster memberships matter, not axis values. UMAP is applied post-hoc for visualisation only; the actual K-Means clustering runs in the full embedding space.

### Architecture note

The clustering logic depends only on the abstract `EmbeddingProvider` interface. Swapping providers requires zero changes to `SemanticClusterer` — this is the Strategy pattern in practice. The auto-detection logic above shows how a single codebase can transparently run local, OpenAI, and Azure backends.